In [8]:
!pip install -q ultralytics opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 9.8 MB/s eta 0:00:00


In [ ]:
import torch
import cv2
import ultralytics

print("YOLO version:", ultralytics.__version__)
print("OpenCV version:", cv2.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
from ultralytics import YOLO

# Loading a small pretrained YOLO model
model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully!")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:


!pip install -q kaggle

import os
import shutil

# Creating a folder for Kaggle credentials
os.makedirs("/root/.kaggle", exist_ok=True)

# Copying the uploaded key into that folder
shutil.copy(
    "/content/kaggle.json",
    "/root/.kaggle/kaggle.json"
)

# Protecting the credentials file
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle configured successfully!")



In [ ]:


!mkdir -p /content/drone_dataset

!kaggle datasets download \
    -d cybersimar08/drone-detection \
    -p /content/drone_dataset \
    --unzip


In [ ]:


import os

dataset_path = "/content/drone_dataset"

print("Dataset folder exists:", os.path.exists(dataset_path))

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)

    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")
        for file in files[:10]:
            print("  " * (level + 1) + file)

In [ ]:

import os

dataset_root = "/content/drone_dataset"

for root, dirs, files in os.walk(dataset_root):
    if "data.yaml" in files:
        print("Found YOLO configuration:")
        print(os.path.join(root, "data.yaml"))


In [ ]:

import yaml

yaml_path = (
    "/content/drone_dataset/"
    "drone-detection-new.v5-new-train.yolov8/data.yaml"
)

with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

print(config)


In [ ]:


import yaml
import os

dataset_dir = (
    "/content/drone_dataset/"
    "drone-detection-new.v5-new-train.yolov8"
)

fixed_config = {
    "path": dataset_dir,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": ["AirPlane", "Drone", "Helicopter"]
}

fixed_yaml = "/content/drone_data.yaml"

with open(fixed_yaml, "w") as f:
    yaml.safe_dump(fixed_config, f, sort_keys=False)

print(open(fixed_yaml).read())


In [ ]:


import os

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(dataset_dir, split, "images")
    label_dir = os.path.join(dataset_dir, split, "labels")

    images = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    labels = [
        f for f in os.listdir(label_dir)
        if f.endswith(".txt")
    ]

    print(f"{split}:")
    print("  Images:", len(images))
    print("  Label files:", len(labels))



In [ ]:

from ultralytics import YOLO

# Loading a small pretrained YOLO model
model = YOLO("yolo11n.pt")

# Training it on the provided drone dataset
results = model.train(
    data=fixed_yaml,
    epochs=5,
    imgsz=640,
    batch=4,
    device="cpu",
    patience=8,
    project="/content/drone_runs",
    name="drone_detection_v2"
)


In [ ]:
from IPython.display import Image, display

display(Image(filename="/content/drone_runs/drone_detection/results.png"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil

source = "/content/drone_runs/drone_detection/weights/best.pt"
destination = "/content/drive/MyDrive/best.pt"

shutil.copy2(source, destination)

print("Model saved:", os.path.exists(destination))
print("Model size:", round(os.path.getsize(destination) / (1024 * 1024), 2), "MB")

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/best.pt")

print("Model loaded successfully!")
print("Classes:", model.names)


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from IPython.display import display, Image
import glob
import os

import os
from IPython.display import display, Image

image_path = next(iter(uploaded.keys()))

results = model.predict(
    source=image_path,
    conf=0.25,
    save=True
)
saved_images = glob.glob("/content/runs/detect/predict/*")

print("Saved files:", saved_images)

if saved_images:
    display(Image(filename=saved_images[0]))

In [ ]:
from google.colab import files

uploaded_video = files.upload()

In [ ]:
import os
from ultralytics import YOLO

# Loading the trained model from Google Drive
model = YOLO("/content/drive/MyDrive/best.pt")

# Get the uploaded video filename
video_path = next(iter(uploaded_video.keys()))

# Tracking objects throughout the video
results = model.track(
    source=video_path,
    conf=0.25,
    tracker="bytetrack.yaml",
    persist=True,
    save=True,
    project="/content/drone_tracking",
    name="tracked_video"
)

print("Tracking finished!")

In [ ]:
import cv2
import pandas as pd
import torch

device = 0 if torch.cuda.is_available() else "cpu"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Error.Could not open video.")

records = []
previous_centres = {}

while True:
    success, frame = cap.read()

    if not success:
        break

    result = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.10,
        classes=[1],  # Drone class in the provided dataset
        device=device,
        verbose=False
    )[0]

    if result.boxes is not None and len(result.boxes) > 0:
        for i, box in enumerate(result.boxes):
            x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()

            centre_x = (x1 + x2) / 2
            centre_y = (y1 + y2) / 2

            track_id = -1
            if box.id is not None:
                track_id = int(box.id.item())

            previous = previous_centres.get(track_id)

            if previous is None:
                change_x = 0
                change_y = 0
            else:
                change_x = centre_x - previous[0]
                change_y = centre_y - previous[1]

            previous_centres[track_id] = (centre_x, centre_y)

            records.append({
                "track_id": track_id,
                "centre_x": centre_x,
                "centre_y": centre_y,
                "change_x": change_x,
                "change_y": change_y
            })

cap.release()

positions = pd.DataFrame(records)
display(positions.head(10))

print("Total tracked detections:", len(positions))

In [ ]:
sensitivity = 0.05

horizontal_angle = 90.0
vertical_angle = 90.0

horizontal_angles = []
vertical_angles = []

for _, row in positions.iterrows():
    horizontal_angle += sensitivity * row["change_x"]
    vertical_angle -= sensitivity * row["change_y"]

    horizontal_angle = max(0, min(180, horizontal_angle))
    vertical_angle = max(0, min(180, vertical_angle))

    horizontal_angles.append(horizontal_angle)
    vertical_angles.append(vertical_angle)

positions["horizontal_servo_angle"] = horizontal_angles
positions["vertical_servo_angle"] = vertical_angles

display(positions.head(10))

In [ ]:
csv_path = "/content/drone_positions.csv"

positions.to_csv(csv_path, index=False)

print("Position and servo data saved to:", csv_path)